# Inference Demo

This notebook demonstrates how to run inference using the **root-level** wrapper scripts you asked for (no changes to `src/`).

- Uses: `config.yaml` + `inference.py`
- Requires: a working FAISS index + chunks in `data/`
- Requires: LLM credentials depending on your chosen backend (Groq/Gemini/Ollama)

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve().parents[0] if (Path.cwd().name == 'notebooks') else Path.cwd().resolve()
print('Repo root:', repo_root)

required_paths = [
    repo_root / 'config.yaml',
    repo_root / 'inference.py',
    repo_root / 'data' / 'chunks.json',
    repo_root / 'data' / 'faiss_index.bin',
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
print('All required files found.')

## Configure backend

Pick a backend by editing `config.yaml` (`backend: groq | gemini | ollama`) or by setting env vars.

Examples (PowerShell):
- Groq: `$env:LLM_BACKEND='groq' ; $env:GROQ_API_KEY='...'`
- Gemini: `$env:LLM_BACKEND='gemini' ; $env:GEMINI_API_KEY='...'`
- Ollama: `$env:LLM_BACKEND='ollama'` (and ensure Ollama is running locally)

In [ ]:
import os

# Optional: set backend here for the notebook session (overrides config.yaml).
# os.environ['LLM_BACKEND'] = 'ollama'
# os.environ['OLLAMA_MODEL'] = 'llama3.1:8b'

# If using Groq:
# os.environ['LLM_BACKEND'] = 'groq'
# os.environ['GROQ_API_KEY'] = '...'

# If using Gemini:
# os.environ['LLM_BACKEND'] = 'gemini'
# os.environ['GEMINI_API_KEY'] = '...'

print('LLM_BACKEND =', os.getenv('LLM_BACKEND'))

## Run inference via `inference.py`

This calls the wrapper script (which imports from `src/` under the hood).

In [ ]:
import subprocess
import sys

query = 'What is retrieval-augmented generation?'
cmd = [sys.executable, str(repo_root / 'inference.py'), '--query', query, '--print-timeline']
print('Running:', ' '.join(cmd))

# Note: This will make LLM calls depending on your backend config.
result = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'inference.py failed with exit code {result.returncode}')